In [ ]:
import requests
import time
from urllib.parse import quote

# BASE = "http://leadere1.local"   # leader stays the API host
# UNIT = "leaderE1"                # <-- change to the worker that has the optics
# EXP  = "testing_testing"   # <-- match the real active experiment name
BASE = "http://leadere1.local"   # leader stays the API host
UNIT = "leaderE1"                # <-- change to the worker that has the optics
EXP  = "overnight_od_via_script"   # <-- match the real active experiment name


# requests.patch(f"{BASE}/api/workers/leaderA1/jobs/run/job_name/od_reading/experiments/{quote(EXP)}",
#                json={"options": {}}, headers={"Content-Type": "application/json"}, timeout=10)
# time.sleep(5)
# requests.patch(f"{BASE}/api/workers/leaderA1/jobs/stop/job_name/od_reading/experiments/{quote(EXP)}",
#                headers={"Content-Type": "application/json"}, timeout=10)

def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok and (job_name not in r.text):
            return True
        time.sleep(poll_s)
    return False

def run_od_snapshot(timeout_s=20):
    """
    Start od_reading with snapshot=True so it takes one reading and exits.
    Assumes od_reading is NOT already running.
    """
    run_url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/od_reading/experiments/{quote(EXP)}"
    payload = {
        "options": {},
        "args": ["--snapshot"]
        }
    resp = requests.patch(run_url, json=payload, headers={"Content-Type": "application/json"})
    print("OD snapshot run:", resp.status_code, getattr(resp, "text", ""))

    done = wait_until_stopped(job_name="od_reading", timeout_s=timeout_s, poll_s=0.5)
    print("OD snapshot status:", "completed" if done else "still running / timed out")
    return resp, done

def is_job_running(job_name="od_reading"):
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    try:
        r = requests.get(url, headers={"Content-Type": "application/json"})
        if r.ok:
            return job_name in r.text
    except Exception:
        pass
    return False

def run_od_continuous():
    """Start od_reading (continuous). Idempotent if already running."""
    if is_job_running("od_reading"):
        print("od_reading already running (continuous).")
        return None
    run_url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/od_reading/experiments/{quote(EXP)}"
    # No snapshot -> continuous according to job defaults
    resp = requests.patch(run_url, json={"options": {}}, headers={"Content-Type": "application/json"})
    print("RUN od_reading (continuous):", resp.status_code, getattr(resp, "text", ""))
    return resp

if __name__ == "__main__":
    try:
        while True:            # run until interrupted
            run_od_snapshot()
            time.sleep(2)      # optional pause between cycles
            # run_od_continuous()
            # time.sleep(2)
    except KeyboardInterrupt:
        print("\nProgram stopped by user (Ctrl-C).")
        # stop_stirring()        # make sure stirring is off at exit

OD snapshot run: 202 {"unit":"leaderA1","task_id":"a4424db8-47c5-4703-ab28-c533e87414e9","result_url_path":"/unit_api/task_results/a4424db8-47c5-4703-ab28-c533e87414e9"}
OD snapshot status: completed
OD snapshot run: 202 {"unit":"leaderA1","task_id":"0fd735d0-72e3-4d9e-a2a4-a8e1fc72fb4e","result_url_path":"/unit_api/task_results/0fd735d0-72e3-4d9e-a2a4-a8e1fc72fb4e"}
OD snapshot status: completed
OD snapshot run: 202 {"unit":"leaderA1","task_id":"22b6fc97-890f-42d8-867a-c41f33a26e3e","result_url_path":"/unit_api/task_results/22b6fc97-890f-42d8-867a-c41f33a26e3e"}
OD snapshot status: completed
OD snapshot run: 202 {"unit":"leaderA1","task_id":"275dd3f7-b4a5-46cf-bf2f-29487ca85399","result_url_path":"/unit_api/task_results/275dd3f7-b4a5-46cf-bf2f-29487ca85399"}
OD snapshot status: completed
OD snapshot run: 202 {"unit":"leaderA1","task_id":"3628f21a-0a95-4d67-b5fa-2503279e105c","result_url_path":"/unit_api/task_results/3628f21a-0a95-4d67-b5fa-2503279e105c"}
OD snapshot status: completed
